# EasyRAG Local Vs Production Backends

## What you will do

- inspect the backend bundle resolver for local and production modes
- compare the class contracts behind those bundles
- run a local backend end to end and verify that the public `EasyRAG` surface does not change

## Why this matters

Backend swaps should change implementation details, not the learning model of the system. This notebook makes that separation explicit.


## Source code anchors

- `easyrag.rag.storage.bundles.resolve_storage_bundle`
- `easyrag.rag.storage.base.BaseKVStorage`
- `easyrag.rag.storage.base.BaseVectorStorage`
- `easyrag.rag.orchestrator.EasyRAG`


In [ ]:
# ruff: noqa: E402,F401,F811
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "easyrag").exists():
        REPO_ROOT = candidate.resolve()
        if str(REPO_ROOT) not in sys.path:
            sys.path.insert(0, str(REPO_ROOT))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

import json
import os
import tempfile
import time
from pathlib import Path
from pprint import pprint
from unittest import mock

from easyrag.rag import AnswerParam, EasyRAG, EvalCase, QueryParam
from easyrag.support.async_utils import run_sync
from notebooks._utils import (
    managed_demo_rag,
    print_json as _print_json,
    production_backends_ready,
    provider_ready as can_use_openai_compatible_models,
    skip_message,
    stub_embedding as _stub_embedding,
    stub_query_model as _stub_query_model,
    stub_reranker as _stub_reranker,
)

from easyrag.rag.storage.base import BaseGraphStorage, BaseKVStorage, BaseVectorStorage
from easyrag.rag.storage.bundles import resolve_storage_bundle


## Deterministic path

We start by asking the resolver which storage classes belong to each backend family.


In [ ]:
local_bundle = resolve_storage_bundle("local")
production_bundle = resolve_storage_bundle("postgres_qdrant")
comparison = {
    "local": [cls.__name__ for cls in local_bundle],
    "postgres_qdrant": [cls.__name__ for cls in production_bundle],
}
_print_json(comparison)


## Output inspection

The next cell verifies that the local classes implement the shared storage contracts and that `EasyRAG` can use the local bundle without changing the public query surface.


In [ ]:
contract_view = {
    "local_vector_is_base": issubclass(local_bundle[1], BaseVectorStorage),
    "local_kv_is_base": issubclass(local_bundle[0], BaseKVStorage),
    "production_vector_is_base": issubclass(production_bundle[1], BaseVectorStorage),
}
_print_json(contract_view)

backend_tmp = tempfile.TemporaryDirectory()
backend_rag = EasyRAG(
    working_dir=backend_tmp.name,
    workspace="backend-demo",
    storage_backend="local",
    embedding_func=_stub_embedding,
    query_model_func=_stub_query_model,
)
run_sync(backend_rag.initialize_storages())
run_sync(backend_rag.ainsert("# Retrieval\nLocal backends still return grounded citations.\n", ids=["doc::retrieval"], file_paths=["docs/retrieval.md"]))
backend_result = run_sync(backend_rag.aquery("grounded citations", QueryParam(mode="naive", rewrite_enabled=False, mqe_enabled=False, chunk_top_k=2)))
_print_json({"metadata": backend_result.metadata, "citations": backend_result.citations})


## Provider-backed path

The optional path is backend-oriented rather than model-oriented here. We only check whether the production bundle can be resolved and whether the environment looks configured enough to attempt it.


In [ ]:
production_ready = bool(os.environ.get("EASYRAG_POSTGRES_DSN")) and bool(os.environ.get("EASYRAG_QDRANT_URL"))
if not production_ready:
    print("Skipping production-backed path because PostgreSQL/Qdrant configuration is not set.")
else:
    try:
        provider_bundle = resolve_storage_bundle("postgres_qdrant")
        print([cls.__name__ for cls in provider_bundle])
    except Exception as exc:
        print(f"Production-backed path was skipped due to runtime error: {exc}")


## What changed and why

The backend bundle changes which storage classes own persistence and similarity search, but the `EasyRAG` lifecycle stays the same: initialize, insert, query, inspect, finalize. That separation is what lets the docs teach one mental model while the deployment backend changes underneath it.


In [ ]:
run_sync(backend_rag.finalize_storages())
backend_tmp.cleanup()
print("Cleaned up the backend comparison workspace.")


## Next steps

- Continue with [08_03_observability_and_fallbacks.ipynb](08_03_observability_and_fallbacks.ipynb) to inspect the runtime signals that tell you when a backend or provider path is degrading.
- Read [engineering/25-local-vs-production-backends.md](../../docs/engineering/25-local-vs-production-backends.md) for the code-oriented backend comparison.
- Revisit [08_01_code_map_and_runtime_flow.ipynb](08_01_code_map_and_runtime_flow.ipynb) if you want the stage-ownership view before backend swaps.
